In [5]:
import os
import sys
import numpy as np
import xarray as xr
from dask.distributed import Client

In [6]:
class E3SMNpyExporter:
    def __init__(
        self,
        topdir="/global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share",
        outdir="/global/cfs/cdirs/e3sm/www/zhan391/SEA_CROGS",
        exp="E3SMv2_NDGUV_tau6",
        groups=None,
        chunks=None,
        start_client=True,
    ):
        self.topdir = topdir
        self.outdir = outdir
        self.exp = exp
        self.groups = groups if groups is not None else ["before_nudging", "nudging_tendency"]
        self.chunks = chunks if chunks is not None else {"time": 100}
        self.client = Client() if start_client else None

        if self.client is not None:
            print("Dask client started:", self.client)

    def get_group_config(self, grp):
        if grp == "before_nudging":
            return {
                "smdh": "01010000",
                "emdh": "12312100",
                "var_list": [
                    "WS_bf_ndg", "WD_bf_ndg", "PRESSURE_bf_ndg",
                    "U_bf_ndg", "V_bf_ndg", "T_bf_ndg",
                    "Q_bf_ndg", "PS_bf_ndg"
                ],
                "var_outs": ["WS", "WD", "PRESSURE", "U", "V", "T", "Q", "PS"],
            }
        else:
            return {
                "smdh": "01010300",
                "emdh": "01010000",
                "var_list": ["Nudge_U", "Nudge_V", "Nudge_T", "Nudge_Q"],
                "var_outs": ["UTEND", "VTEND", "TEND", "QTEND"],
            }

    def build_input_file(self, grp, year, var, smdh, emdh):
        datadir = os.path.join(self.topdir, "SE_PG2", grp)

        if "Nudge" in var:
            fname = f"{self.exp}_3hourly_{year:04d}{smdh}-{year + 1:04d}{emdh}.nc"
        else:
            fname = f"{self.exp}_3hourly_{year:04d}{smdh}-{year:04d}{emdh}.nc"

        return os.path.join(datadir, fname)

    def build_output_file(self, vou, year):
        outpath = os.path.join(self.outdir, self.exp)
        os.makedirs(outpath, exist_ok=True)
        outname = f"{vou}_3hourly_{year:04d}.npy"
        return os.path.join(outpath, outname)

    def print_dataset_info(self, ds):
        lat = ds["lat"].values
        lon = ds["lon"].values
        lev = ds["lev"].values
        time = ds["time"].values

        print(f"latitude range: {lat.min()} to {lat.max()}")
        print(f"longitude range: {lon.min()} to {lon.max()}")
        print(f"level range: {lev.min()} to {lev.max()}")
        print(f"time range: {time.min()} to {time.max()}")

    def extract_variable(self, ds, var):
        if var == "PRESSURE_bf_ndg":
            hyam = ds["hyam"].values
            hybm = ds["hybm"].values
            P0 = ds["P0"].values
            PS = ds["PS_bf_ndg"]

            pressure = []
            for k in range(len(ds["lev"].values)):
                pressure.append(hyam[k] * P0 + hybm[k] * PS)

            data = xr.concat(pressure, dim="lev")
            data_np = data.transpose("time", "lev", "lat").compute().astype(np.float32)

        elif var == "PS_bf_ndg":
            data_np = ds[var].compute().astype(np.float32)

        elif var in ["WS_bf_ndg", "WD_bf_ndg"]:
            U = ds["U_bf_ndg"]
            V = ds["V_bf_ndg"]
            WS = (U**2 + V**2) ** 0.5
            WD = np.arctan2(V, U)
            data = WS if var == "WS_bf_ndg" else WD
            data_np = data.compute().astype(np.float32)

        else:
            data_np = ds[var].compute().astype(np.float32)

        return data_np

    def process_variable(self, grp, year, var, vou, smdh, emdh):
        print(f"\nProcessing variable: {var}")

        fpath = self.build_input_file(grp, year, var, smdh, emdh)
        if not os.path.exists(fpath):
            print(f"File not found: {fpath}")
            return

        try:
            ds = xr.open_dataset(fpath, chunks=self.chunks)
        except Exception as e:
            print(f"Failed to open {os.path.basename(fpath)}: {e}")
            return

        try:
            self.print_dataset_info(ds)
            data_np = self.extract_variable(ds, var)
        except Exception as e:
            print(f"Error reading variable {var}: {e}")
            ds.close()
            return

        fout = self.build_output_file(vou, year)

        if os.path.exists(fout):
            os.remove(fout)

        np.save(fout, data_np)
        print(f"Saved: {fout}")

        ds.close()
        del data_np

    def run(self, syear, eyear):
        for grp in self.groups:
            cfg = self.get_group_config(grp)
            smdh = cfg["smdh"]
            emdh = cfg["emdh"]
            var_list = cfg["var_list"]
            var_outs = cfg["var_outs"]

            for year in range(syear, eyear + 1):
                for var, vou in zip(var_list, var_outs):
                    self.process_variable(grp, year, var, vou, smdh, emdh)


In [7]:
if __name__ == "__main__":
    topdir="/global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share"
    outdir="/global/cfs/cdirs/e3sm/zhan391/SEA_CROGS/New_Training_2025"
    exp="E3SMv2_NDGUV_tau6"
    syear = 2007
    eyear = 2017
    exporter = E3SMNpyExporter()
    exporter.run(syear, eyear)
    

/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37199 instead
  warnings.warn(


Dask client started: <Client: 'tcp://127.0.0.1:33123' processes=16 threads=256, memory=502.97 GiB>

Processing variable: WS_bf_ndg
File not found: /global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share/SE_PG2/before_nudging/E3SMv2_NDGUV_tau6_3hourly_200701010000-200712312100.nc

Processing variable: WD_bf_ndg
File not found: /global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share/SE_PG2/before_nudging/E3SMv2_NDGUV_tau6_3hourly_200701010000-200712312100.nc

Processing variable: PRESSURE_bf_ndg
File not found: /global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share/SE_PG2/before_nudging/E3SMv2_NDGUV_tau6_3hourly_200701010000-200712312100.nc

Processing variable: U_bf_ndg
File not found: /global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share/SE_PG2/before_nudging/E3SMv2_NDGUV_tau6_3hourly_200701010000-200712312100.nc

Processing variable: V_bf_ndg
File not found: /global/cfs/cdirs/e3sm/www/zhan391/darpa_temporary_data_share/SE_PG2/before_nudging/E3SMv2_NDGUV_tau6